# 🔄 vLLM Prefix Caching (APC) — 自动前缀缓存深度解析

**本文目标**：深入理解 vLLM 的 Automatic Prefix Caching (APC) 的工作原理、数据结构和配置策略。

读完这篇你会理解：
- APC 的 hash-based 匹配机制
- Block-level 前缀共享 vs Token-level (SGLang) 的差异
- APC 在工具调用和多模态场景的实际收益
- 如何监控和调优 APC 命中率

## 1. APC 的工作原理

### 1.1 Block Hash → 缓存命中

```python
# vLLM APC 的核心流程 (简化)

class PrefixCachingBlock:
    """支持前缀缓存的 Block"""
    
    def __init__(self, block_id, token_ids, kv_cache_data):
        self.block_id = block_id
        self.token_ids = tuple(token_ids)  # 用于 hash 计算
        self.kv_cache = kv_cache_data
        self._hash = hash(self.token_ids)
        self.ref_count = 0
    
    @property
    def content_hash(self):
        """用于缓存匹配的 hash"""
        return self._hash

# 全局缓存
_prefix_cache: Dict[int, PrefixCachingBlock] = {}

def get_or_compute_block(token_ids, compute_fn):
    """获取或计算一个 block 的 KV Cache"""
    h = hash(tuple(token_ids))
    
    if h in _prefix_cache:
        # 缓存命中! → 直接返回, 跳过 attention 计算
        block = _prefix_cache[h]
        block.ref_count += 1
        return block, hit=True
    else:
        # 缓存未命中 → 计算 attention, 存入缓存
        kv_data = compute_fn(token_ids)
        block = PrefixCachingBlock(assign_id(), token_ids, kv_data)
        _prefix_cache[h] = block
        return block, hit=False
```

### 1.2 命中条件: 严格的 block 边界对齐

```
APC 命中需要:
  1. Token 序列完全一致
  2. Block 边界完全对齐

示例 (block_size=16):

请求 A (token 0-15):  "You are a helpful assistant..."  (16 tokens)
                        → Block hash: 0xABCD → 缓存

请求 B (token 0-15):  "You are a helpful assistant..."  (完全相同)
                        → Block hash: 0xABCD → 命中! ✓

请求 C (token 0-15):  "You are a helpful assistant. You"  (16 tokens)
                        → Block hash: 0x1234 → 未命中 ✗
                        (多了一个 "You", block 内容不同)

请求 D (token 0-15):  "You are a helpful assistant..."  ← 同 A/B
    token 16-17: "Please"  (2 tokens, 不足一个 block)
    → Block 0 hash: 0xABCD → 命中! ✓ (Block 0 完全相同)
    → Block 1 还没满 → 等攒够 16 个 token 再 hash → 未命中

关键限制:
  - 必须从第一个 block 开始才能命中
  - 每个 block 必须 token 完全一致才能匹配
  - Block 边界对齐: 即使 token 序列相同, 如果分割方式不同 → 也 miss
```

### 1.3 与 SGLang RadixAttention 的对比

```
vLLM APC (block-level hash):
  [Block0: "You are a helpful..."] → hash → 匹配/不匹配
  ↑ 粒度: block (16 tokens)

SGLang RadixAttention (token-level radix tree):
  ["You"] → ["are"] → ["a"] → ["helpful"] → ...
  ↑ 粒度: token

场景: 两个请求, system prompt 的前 15 个 token 相同,
      但第 16 个 token 不同

vLLM APC:
  Block 0 包含 token 0-15
  请求 A: [0-15 为 system prompt]
  请求 B: [0-14 为 system prompt] [15 为新 token]
  → Block 0 内容不同 → MISS ✗

SGLang RadixAttention:
  Token 0-14 在 radix tree 中形成共享路径
  → Token 0-14 命中! ✓
  → Token 15 分叉 → 各自建立新路径

结论: SGLang 的粒度优势在 "接近但不完全一致" 的前缀场景中明显
```

## 2. APC 的显存管理

### 2.1 Eviction 策略

```python
# 当显存不足时, APC 需要淘汰缓存的 block

# 策略 1: LRU (Least Recently Used)
# 淘汰最久未使用的 block

# 策略 2: Reference-count based
# 当 block 的引用计数降为 0 → 自动回收
# 如果有 Sequence 正在使用 → 保留

# vLLM 实际使用的是混合策略:
# - 每个 block 有 ref_count (被多少 Sequence 引用)
# - ref_count = 0 → 可回收
# - 回收时按 LRU 排序: 同 ref_count 的 block, 淘汰最旧的
```

### 2.2 APC 的显存开销

```
APC 缓存本身也占用显存!

每个缓存的 block:
  KV Cache 数据: block_size × 2 × n_kv_heads × head_dim × dtype
  例如: 16 × 2 × 8 × 128 × 2 bytes (FP16) = 65 KB per block

  Hash 表 overhead: ~1% of cached data

1000 个缓存 block → ~65 MB
10,000 个缓存 block → ~650 MB
→ 可能需要 --gpu-memory-utilization 调低一点给 cache 留空间
```

## 3. 实战: APC 在工具调用中的配置

```bash
# 开启 APC
vllm serve meta-llama/Llama-3-8B-Instruct     --enable-prefix-caching     --gpu-memory-utilization 0.88  # 留空间给 APC cache

# 检查 APC 命中率 (通过 metrics endpoint)
curl http://localhost:8000/metrics | grep prefix_cache
# vllm:gpu_prefix_cache_queries_total     # 查询次数
# vllm:gpu_prefix_cache_hits_total        # 命中次数
# hit_rate = hits / queries

# 工具调用场景优化:
# 1. 统一 function definitions 的格式
# 2. system prompt 放在最前面 (便于 prefix 命中)
# 3. 不要在不同请求间改变 system prompt 的 token 序列
```

### 3.1 命中率预期

```
场景                 命中率    说明
纯文本对话           ~5%      system prompt 可能部分命中
Agent (相同工具集)    ~20-30%  system prompt + tool call 格式命中
批量处理相同 prompt   ~80%+   几乎整个 prompt 命中
多模态 (同图)         ~30%    visual tokens 可能命中 (如果同图)

APC 不是银弹, 但 0 成本 (不需要改动代码), 建议始终开启
```

In [ ]:
# APC 缓存命中率模拟

import random

def simulate_apc(requests, block_size=16):
    """模拟 APC 在 N 个请求下的表现"""
    hits = 0
    misses = 0
    cache = {}  # hash → block
    
    for req_idx, tokens in enumerate(requests):
        for i in range(0, len(tokens), block_size):
            block = tuple(tokens[i:i+block_size])
            if len(block) < block_size:
                break  # 不足一个 block, 不参与缓存
            h = hash(block)
            if h in cache:
                hits += 1
            else:
                cache[h] = True
                misses += 1
    
    return hits, misses

# 场景 1: Agent 请求 (system prompt 相同, tool results 不同)
print("=" * 60)
print("APC 命中率模拟")
print("=" * 60)

n_requests = 50
system_prompt = list(range(2000))  # system prompt (所有请求共享)

agent_requests = []
for _ in range(n_requests):
    req = system_prompt.copy()
    # 每轮: user + tool_call + random result + assistant
    for r in range(5):
        req.append(10000 + random.randint(0, 999))  # user (简化为 1 token)
        req.extend([30000] * 30)  # tool_call (固定格式)
        req.extend([40000 + random.randint(0, 9999) for _ in range(1000)])  # result (随机)
        req.extend([50000 + random.randint(0, 999) for _ in range(100)])  # assistant (随机)
    agent_requests.append(req)

h, m = simulate_apc(agent_requests)
print(f"\nAgent 场景 ({n_requests} 请求, 5 轮工具调用):")
print(f"  总 block 查询: {h+m}, 命中: {h}, 未命中: {m}")
print(f"  命中率: {h/(h+m)*100:.1f}%")
print(f"  分析: system prompt (~2K tokens) 共享, tool_call (~150 tokens) 部分共享")
print(f"        tool_result (~5K tokens) 每次不同 → 限制命中率")

# 场景 2: 多模态 (同图)
visual_req = []
for _ in range(n_requests):
    req = list(range(576))  # 同一张图的 visual tokens
    req.extend([10000 + random.randint(0, 9999) for _ in range(100)])  # 不同文本
    visual_req.append(req)

h2, m2 = simulate_apc(visual_req)
print(f"\n多模态场景 ({n_requests} 请求, 同图):")
print(f"  命中率: {h2/(h2+m2)*100:.1f}%")
print(f"  分析: 同图的 visual tokens 完美命中, 不同文本部分 miss")

# 场景 3: 纯文本 (system prompt 不同 → 预期低命中)
text_requests = []
for _ in range(n_requests):
    req = [random.randint(0, 9999) for _ in range(random.randint(100, 500))]
    text_requests.append(req)

h3, m3 = simulate_apc(text_requests)
print(f"\n纯文本场景 ({n_requests} 请求, 各不相同):")
print(f"  命中率: {h3/(h3+m3)*100:.1f}%")
print(f"  分析: 随机内容 → 几乎无命中 (预期 ~0%)")